In [28]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from simulations import simulations as sim

# cerebellar ROIs
df1 = sim.y_sim()
df2 = sim.y_sim(seed = 9, region = 'region2')
df3 = sim.y_sim(seed = 99, region = 'region3')
df = pd.concat([df1, df2, df3], axis=0)

# tract ROIs
r1 = sim.y_sim(seed = 1, region = 'tract1')
r2 = sim.y_sim(seed = 2, region = 'tract2')
df = pd.concat([df, r1, r2], axis=0)

In [29]:
def _predict_roi_df(response_df, predict_df):
   # this will give a dataframe for each response_roi-week, you have vlaues of all weeks for each pred_roi
   predictors_df = predict_df.rename(columns = {'regionname': 'pred_region', 'mean': 'pred_mean', 'Week': 'pred_week'})
   roi_df = response_df.merge(predictors_df, on = 'subj_id', how = 'inner')


   return roi_df


response_df = pd.concat([df1, df2, df3], axis = 0)
predict_df = pd.concat([r1, r2], axis = 0)

roi_df = _predict_roi_df(response_df, predict_df)

In [30]:
models = {}
for pred_region in roi_df.pred_region.unique():
    for response_region in roi_df.regionname.unique():
        region_df = roi_df[(roi_df.pred_region == pred_region) & (roi_df.regionname == response_region)]
        model = smf.mixedlm("mean~0+ C(Week)*C(pred_week)*pred_mean", data = region_df, groups = region_df.subj_id).fit(maxiter = 400)
        models[(pred_region, response_region)] = model

In [31]:
models[(pred_region, response_region)].summary()

<class 'statsmodels.iolib.summary2.Summary'>
"""
                        Mixed Linear Model Regression Results
======================================================================================
Model:                       MixedLM           Dependent Variable:           mean     
No. Observations:            500               Method:                       REML     
No. Groups:                  20                Scale:                        0.0000   
Min. group size:             25                Log-Likelihood:               4374.1793
Max. group size:             25                Converged:                    Yes      
Mean group size:             25.0                                                     
--------------------------------------------------------------------------------------
                                           Coef.  Std.Err.    z    P>|z| [0.025 0.975]
--------------------------------------------------------------------------------------
C(Week)[0]                                  6.946    0.367  18.951 0.000  6.228  7.664
C(Week)[4]                                 10.946    0.367  29.863 0.000 10.228 11.664
C(Week)[12]                                18.946    0.366  51.699 0.000 18.228 19.664
C(Week)[24]                                30.946    0.367  84.435 0.000 30.228 31.664
C(Week)[52]                                58.946    0.367 160.810 0.000 58.228 59.665
C(pred_week)[T.4]                           0.338    0.248   1.363 0.173 -0.148  0.823
C(pred_week)[T.12]                          1.013    0.743   1.363 0.173 -0.443  2.469
C(pred_week)[T.24]                          2.025    1.486   1.363 0.173 -0.887  4.937
C(pred_week)[T.52]                          4.388    3.219   1.363 0.173 -1.922 10.698
C(Week)[T.4]:C(pred_week)[T.4]             -0.000    0.000  -0.000 1.000 -0.000  0.000
C(Week)[T.12]:C(pred_week)[T.4]             0.000    0.000   0.000 1.000 -0.000  0.000
C(Week)[T.24]:C(pred_week)[T.4]            -0.000    0.000  -0.000 1.000 -0.000  0.000
C(Week)[T.52]:C(pred_week)[T.4]             0.000    0.000   0.000 1.000 -0.000  0.000
C(Week)[T.4]:C(pred_week)[T.12]             0.000    0.000   0.000 1.000 -0.000  0.000
C(Week)[T.12]:C(pred_week)[T.12]            0.000    0.000   0.000 1.000 -0.000  0.000
C(Week)[T.24]:C(pred_week)[T.12]            0.000    0.000   0.000 1.000 -0.000  0.000
C(Week)[T.52]:C(pred_week)[T.12]            0.000    0.000   0.000 1.000 -0.000  0.000
C(Week)[T.4]:C(pred_week)[T.24]             0.000    0.000   0.000 1.000 -0.000  0.000
C(Week)[T.12]:C(pred_week)[T.24]            0.000    0.000   0.000 1.000 -0.000  0.000
C(Week)[T.24]:C(pred_week)[T.24]            0.000    0.000   0.000 1.000 -0.000  0.000
C(Week)[T.52]:C(pred_week)[T.24]            0.000    0.000   0.000 1.000 -0.000  0.000
C(Week)[T.4]:C(pred_week)[T.52]            -0.000    0.000  -0.000 1.000 -0.000  0.000
C(Week)[T.12]:C(pred_week)[T.52]            0.000    0.000   0.000 1.000 -0.000  0.000
C(Week)[T.24]:C(pred_week)[T.52]           -0.000    0.000  -0.000 1.000 -0.000  0.000
C(Week)[T.52]:C(pred_week)[T.52]           -0.000    0.000  -0.000 1.000 -0.000  0.000
pred_mean                                  -0.084    0.062  -1.363 0.173 -0.206  0.037
C(Week)[T.4]:pred_mean                     -0.000    0.000  -0.000 1.000 -0.000  0.000
C(Week)[T.12]:pred_mean                    -0.000    0.000  -0.000 1.000 -0.000  0.000
C(Week)[T.24]:pred_mean                    -0.000    0.000  -0.000 1.000 -0.000  0.000
C(Week)[T.52]:pred_mean                     0.000    0.000   0.000 1.000 -0.000  0.000
C(pred_week)[T.4]:pred_mean                -0.000    0.000  -0.000 1.000 -0.000  0.000
C(pred_week)[T.12]:pred_mean               -0.000    0.000  -0.000 1.000 -0.000  0.000
C(pred_week)[T.24]:pred_mean               -0.000    0.000  -0.000 1.000 -0.000  0.000
C(pred_week)[T.52]:pred_mean               -0.000    0.000  -0.000 1.000 -0.000  0.000
C(Week)[T.4]:C(pred_week)[T.4]:pred_mean    0.000    0.000   